# CEDAR Query API - Example Search Scenarios and Queries

The Cancer Epitope Database and Analysis Resource (CEDAR) API (Application Programming Interface) enables users to perform queries and retrieve CEDAR data programmatically within their preferred environment. This Python Notebook Series illustrates basic usage examples of the CEDAR API using Python.

The CEDAR API is built on a PostgREST platform, providing transparent access to the tables on the backend. For more information on the expressive syntax of PostgREST, refer to this [document](https://postgrest.org/en/stable/api.html#). The CEDAR data can be queried through search and export endpoints described in the [CEDAR API documentation](https://discuss.iedb.org/t/application-programming-interface-api/220) and the interactive [Swagger documentation](https://cedar-api.iedb.org/docs/swagger/).

These example search scenarios and queries are derived from this [Book Chapter](https://doi.org/10.1007/978-1-0716-4566-6_3), which describes how to perform different queries to the [CEDAR homepage](https://cedar.iedb.org). This notebook reproduces the same queries using the CEDAR API.

## Research scenario I

For a literature review, we want to retrieve all the references that contain assays testing epitopes from the antigen “Prostate-specific antigen”.

### Approach

To retrieve this data, we will execute the following steps:
1. **Search references** - Identify the articles that report assays of the target antigen
3. **Retrieve complete reference data** - Obtain full bibliographic information

Each step requires a request to a different interconnected endpoint. For more information, refer to the [PostgREST resource embedding documentation](https://docs.postgrest.org/en/v12/references/api/resource_embedding.html#resource-embedding).

### Setup

First, import the required modules and define a function to make CEDAR API requests. This function takes the endpoint, the search parameters, and the base URI (Unified Resource Identifier) as inputs. Based on these, it constructs the URL (Unified Resource Locator) to interact with the data and resources specified. Then, it performs the API request.
Since the API has a maximum limit of 10,000 rows per request, the function iterates through results, retrieving data in chunks.
Finally, it parses the response and returns a pandas DataFrame. For more information, refer to [pandas documentation](https://pandas.pydata.org/).


In [1]:
import os
import requests
import pandas as pd
import io
import time

pd.set_option('display.max_columns', 100)

def API_query(endpoint, query_params, base_uri='https://cedar-api.iedb.org/'):
    """
    Execute a query against the CEDAR API with pagination support.

    Parameters:
    -----------
    endpoint : str
        The API endpoint to query
    query_params : dict
        Dictionary of query parameters
    base_uri : str
        Base URI for the CEDAR API

    Returns:
    --------
    pd.DataFrame
        Combined results from all paginated requests
    """

    url = os.path.join(base_uri, endpoint)
    df = pd.DataFrame()

    # set the offset to 0
    query_params['offset'] = 0

    # loop through the pages of results
    # API only allows pulling 10,000 entries at a time
    while(True):
        print('Fetching offset: %i' % query_params['offset'])
        r = requests.get(
            url,
            params=query_params,
            headers={'accept': 'text/csv', 'Prefer': 'count=exact'}
        )

        try:
            # Parse CSV response and append to existing DataFrame
            df = pd.concat([df, pd.read_csv(io.StringIO(r.content.decode('utf-8')))])
            query_params['offset'] += 10000
        except pd.errors.EmptyDataError:
            # No more data available
            break

        # Rate limiting: pause between requests to not overload the server
        time.sleep(1)

    return df


### Step 1: Search Epitopes

In this example, we are searching for **references** that tested a specific antigen.

The CEDAR API provides two types of endpoints, the search and export. The search endpoints contain fields with information that facilitate to programatically identify and/or filter the data of interest, while the export endpoints organize the data in a user-friendly format, matching the structure and naming conventions of the custom exports on the CEDAR website.

To perform our query, we will first use the `reference_search` endpoint. Let's examine its structure and available fields.

In [2]:
cedar_url='https://cedar-api.iedb.org/'
endpoint='reference_search'

table = pd.json_normalize(requests.get(os.path.join(cedar_url, endpoint + '?limit=3')).json())
display(table)

,reference_id,reference_iri,e_modifications,linear_sequence_lengths,qualitative_measures,journal_name,antibody_isotypes,direct_ex_vivo_bool,submission_ids,submission_iris,pdb_ids,cedar_assay_ids,cedar_assay_iris,structure_ids,structure_iris,structure_types,linear_sequences,mhc_classes,mhc_allele_resolutions,parent_source_antigen_iris,parent_source_antigen_names,parent_source_antigen_source_org_iris,parent_source_antigen_source_org_names,e_related_object_types,tcell_ids,tcell_iris,bcell_ids,bcell_iris,elution_ids,elution_iris,receptor_ids,receptor_group_ids,tcr_receptor_group_ids,bcr_receptor_group_ids,receptor_group_iris,tcr_receptor_group_iris,bcr_receptor_group_iris,receptor_types,receptor_names,receptor_chain1_types,receptor_chain2_types,receptor_chain1_cdr1_seqs,receptor_chain1_cdr2_seqs,receptor_chain1_cdr3_seqs,receptor_chain1_full_seqs,receptor_chain2_cdr1_seqs,receptor_chain2_cdr2_seqs,receptor_chain2_cdr3_seqs,receptor_chain2_full_seqs,reference_title,reference_title2,reference_author,reference_author2,reference_date,reference_date2,pubmed_id,mhc_allele_iri_search,mhc_allele_iris,mhc_allele_names,assay_iri_search,assay_iris,assay_names,epitope_structures_defined,host_organism_iri_search,host_organism_iris,host_organism_names,source_organism_iri_search,source_organism_iris,source_organism_names,disease_iri_search,disease_iris,disease_names,parent_source_antigen_iri_search,non_peptidic_molecule_iri_search,non_peptidic_molecule_iris,non_peptidic_molecule_names,r_object_source_molecule_iri_search,r_object_source_molecule_iris,r_object_source_molecule_names,r_object_source_organism_iri_search,r_object_source_organism_iris,r_object_source_organism_names,mhc_allele_evidences,chebi_ids,structure_descriptions,curated_source_antigens,neoantigen_bool,viral_antigen_bool,germline_antigen_bool,other_antigen_bool,disease_stages,naturally_occuring_disease_bool,animal_model_of_cancer_bool,all_vaccination_bool,prophylactic_vaccination_bool,therapeutic_vaccination_bool,mutations
0,1451,CEDAR_REFERENCE:1451,None,[10],"[Negative, Positive]",Nat Med,None,1,None,None,None,"[22636, 22637, 22667, 22668, 22671, 22672, 226...","[CEDAR_ASSAY:22636, CEDAR_ASSAY:22637, CEDAR_A...","[9489, 13487, 13497, 13603, 15822, 17162, 1716...","[CEDAR_EPITOPE:13487, CEDAR_EPITOPE:13497, CED...",[Linear peptide],"[DNITSSVLFN, ENIKKILLRE, ENIYYSSVRT, ENVKKSRRL...",[II],[1 chain],"[UNIPROT:H7C7K8, UNIPROT:O50846, UNIPROT:O5138...",[C-C chemokine receptor type 7 (UniProt:P32248...,"[NCBITaxon:139, NCBITaxon:3050296, NCBITaxon:3...",[Borreliella burgdorferi (Lyme disease spiroch...,None,"[22636, 22637, 22667, 22668, 22671, 22672, 226...","[CEDAR_ASSAY:22636, CEDAR_ASSAY:22637, CEDAR_A...",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,Identification of candidate T-cell epitopes an...,None,B Hemmer; B Gran; Y Zhao; A Marques; J Pascal;...,None,1999,None,10581079,"[GO:0032991, GO:0042611, MRO:0000011, MRO:0000...",[MRO:0001331],[HLA-DRB1*15:01],"[OBI:0002055, OBI:0002071, OBI:0003128, OBI:11...",[OBI:1110180],[proliferation|3H-thymidine],[Exact Epitope],"[NCBITaxon:314295, NCBITaxon:40674, NCBITaxon:...",[taxon:10000119],[Homo sapiens Caucasian],"[NCBITaxon:1, NCBITaxon:10239, NCBITaxon:10370...","[NCBITaxon:10370, NCBITaxon:11676, NCBITaxon:1...",[Borreliella burgdorferi (Lyme disease spiroch...,"[DOID:0050117, DOID:104, DOID:11729, DOID:4, O...",[DOID:11729],[Lyme disease],"[PR:000000001, taxon_protein:10002039, taxon_p...",None,None,None,None,None,None,None,None,None,[Not determined],None,"[DNITSSVLFN, ENIKKILLRE, ENIYYSSVRT, ENVKKSRRL...","[{'accession': 'AAB06256.1', 'name': 'envelope...",0,0,1,1,[Chronic],1,0,0,0,0,None
1,1045896,CEDAR_REFERENCE:1045896,None,"[15, 21]","[Positive, Positive-Low]",PLoS One,None,1,None,None,None,"[23044595, 23044596, 23044597, 23044598, 23044...","[CEDAR_ASSAY:23044595, CEDAR_ASSAY:23044596, C...","[2280959, 2280960, 2281028]","[CEDAR_EPITOPE:2280959, CEDAR_EPITO

Next, we define the search parameters, which consist of three components:

- The **query filters:**  that we want to impose on the data.

  Here, we are interested in epitopes derived from a specific source antigen. It is possible to filter by the protein name using the `parent_source_antigen_name` field (e.g. `ilike.*Prostate-specific antigen*`). However, for more efficient and exact results, it is preferred to search by a protein identifier using the `parent_source_antigen_iri_search` field. This field contains a tree of IRIs (Internationalized Resource Identifiers) that point to external resources such as [UniProt](www.uniprot.org), allowing users to filter the data using stable and unique identifiers. The UniProt ID of the Prostate-specific antigen is [P07288](https://www.uniprot.org/uniprotkb/P07288/entry).

  
  We utilize operators in the query parameters, such as `ilike` and `ov`. These operators allow us to perform flexible queries and refine the search. For more information on the operators' syntax, refer to this [documentation](https://docs.postgrest.org/en/v12/references/api/tables_views.html#operators).

- The **pagination parameters** required to retrieve the results in chunks. The order parameter is a field used to sort and paginate the data so that each retrieved chunk contains different consecutive rows. The offset parameter indicates the index of the first element of each data chunk.
- The final **column selection**. In this case, we will keep the `reference_id` field. It will be used for subsequent queries to other endpoints.

In [3]:
cedar_url='https://cedar-api.iedb.org/'
endpoint='reference_search'

search_params={
    # Query filters
    'parent_source_antigen_iri_search':'ov.{UNIPROT:P07288}',

    # Pagination
    'order': 'reference_id',
    'offset': 0,

    # Column selection
    'select': 'reference_id',
}

# Execute reference search
reference_df = API_query(endpoint, search_params)

print(f"Retrieved {len(reference_df)} reference records")
reference_df.head()

Fetching offset: 0
Fetching offset: 10000
Retrieved 85 reference records


,reference_id
0,305
1,200059
2,200060
3,200080
4,315269


### Step 2: Retrieve Complete Reference Data

With the reference IDs, we query the `reference_export` endpoint. This table contains the complete bibliographic information associated to the publications. To perform this query, we format the reference IDs, include them in the search parameters, and perform the API request.

In [4]:
endpoint='reference_export'

# Format unique reference IDs
reference_ids = ','.join(map(str, set(reference_df['reference_id'].tolist())))

search_params = {
    # Query filter
    'reference_id': f'in.({reference_ids})',

    # Pagination
    'order': 'reference_id',
    'offset': 0,
}

# Execute final query
references_df = API_query(endpoint, search_params)

print(f"Retrieved {len(references_df)} reference records")
references_df.head()

Fetching offset: 0
Fetching offset: 10000
Retrieved 85 reference records


,reference_id,reference_id__cedar_iri,reference__type,reference__pmid,reference__submission_id,reference__alternate_iris,reference__authors,reference__journal,reference__date,reference__title
0,305,https://cedar.iedb.org/reference/305,Literature,15651071.0,NaN,NaN,Maxim Pavlenko; Christoph Leder; Anna-Karin Ro...,The Prostate,2005,Identification of an immunodominant H-2D(b)-re...
1,200059,https://cedar.iedb.org/reference/200059,Literature,8871647.0,NaN,NaN,J Sidney; S Southwood; M F del Guercio; H M Gr...,"Journal of immunology (Baltimore, Md. : 1950)",1996,Specificity and degeneracy in peptide binding ...
2,200060,https://cedar.iedb.org/reference/200060,Literature,8882405.0,NaN,NaN,J Sidney; H M Grey; S Southwood; E Celis; P A ...,Human immunology,1996,Definition of an HLA-A3-like supermotif demons...
3,200080,https://cedar.iedb.org/reference/200080,Literature,9438205.0,NaN,NaN,J Sidney; M F del Guercio; S Southwood; G Herm...,Human immunology,1997,The HLA-A*0207 peptide binding repertoire is l...
4,315269,https://cedar.iedb.org/reference/315269,Literature,9743387.0,NaN,NaN,P Correale; K Walmsley; S Zaremba; M Zhu; J Sc...,"Journal of immunology (Baltimore, Md. : 1950)",1998,Generation of human cytolytic T lymphocyte lin...


### Results

The retrieved data contains all publications reporting T cell assays of epitopes from the Prostate-specific antigen. This programmatic approach demonstrates the CEDAR API's capability for automated data retrieval and analysis.

## References

Koşaloğlu-Yalçın, Z. et al. (2025) *Using the Cancer Epitope Database and Analysis Resource (CEDAR)*. Alexander Krasnitz and Pascal Belleau (Eds.), *Cancer Bioinformatics, Methods in Molecular Biology*, vol. 2932. [https://doi.org/10.1007/978-1-0716-4566-6_3 ](https://doi.org/10.1007/978-1-0716-4566-6_3)

Koşaloğlu-Yalçın, Z. et al. (2023) *The cancer epitope database and analysis resource (CEDAR).* Nucleic acids research 51, no. D1 (2023): D845-D852. [https://doi.org/10.1093/nar/gkac902 ](https://doi.org/10.1093/nar/gkac902)